1. load the dataset

In [1]:
# ============================================================================
# COMPLETE TABTRANSFORMER PIPELINE - FROM DATA LOADING TO MODEL TRAINING
# Each section is a separate cell - copy into your Jupyter notebook
# ============================================================================


"""
================================================================================
CELL 1: Install Required Libraries
================================================================================
"""
!pip install boto3
!pip install pandas
!pip install tensorflow
!pip install scikit-learn

print("✓ All packages installed!")


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
✓ All packages installed!


In [2]:
"""
================================================================================
CELL 2: Load Dataset from RunPod S3
================================================================================
"""
import boto3
import pandas as pd
from botocore.config import Config
from io import BytesIO

print("Loading dataset from RunPod S3...")

# ---- RunPod S3 location ----
BUCKET = "e9tcw5eupu"
KEY = "data/eff_testingA.csv"
ENDPOINT = "https://s3api-eu-ro-1.runpod.io"
REGION = "eu-ro-1"

# ---- Your RunPod S3 credentials ----
ACCESS_KEY = "user_37sKcYrvnk9UXaIY3B3Zr90MH0g"
SECRET_KEY = "rps_YW72UMRXEMRVC8A407OCL08J8G34U1B3QTNO1ETX18pa1n"

cfg = Config(
    region_name=REGION,
    signature_version="s3v4",
    s3={"addressing_style": "path"},
)

s3 = boto3.client(
    "s3",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    endpoint_url=ENDPOINT,
    config=cfg,
)

# Load CSV from RunPod S3 into a variable named `data`
obj = s3.get_object(Bucket=BUCKET, Key=KEY)
data = pd.read_csv(BytesIO(obj["Body"].read()))

print(f"✓ Data loaded successfully!")
print(f"  Data type: {type(data)}")
print(f"  Data shape: {data.shape}")
print(f"\nFirst few rows:")
print(data.head())

Loading dataset from RunPod S3...
✓ Data loaded successfully!
  Data type: <class 'pandas.DataFrame'>
  Data shape: (1684, 5138)

First few rows:
                           photo_id        f1        f2        f3        f4  \
0  e5ae8fe5bbdf611a1e8d06e66e849bdf  0.073159  0.085775 -0.133776  0.881202   
1  605a5fd09058c48156b0ef518b63b2de  0.092031 -0.066016 -0.145132  0.687441   
2  909c9277309e13ee014e347603aba620  0.057046 -0.051366 -0.148253  0.675916   
3  bef6a68bc8dd475c124f6de2413385d3 -0.018792  0.016435 -0.148091  0.464433   
4  6d7ed4bc4a17546447efed0ca6e2ff11  0.084419  0.065945 -0.153379  0.635377   

         f5        f6        f7        f8        f9  ...         hip  \
0  0.214236  0.016104 -0.180302 -0.100713 -0.117249  ...  106.774690   
1  0.186508 -0.075221 -0.093846 -0.035840  0.033903  ...  102.481633   
2  0.209973 -0.073485 -0.072783 -0.059395  0.008370  ...   99.342301   
3  0.242849 -0.106556  0.001489 -0.083478  0.096048  ...  101.770144   
4  0.285274 -0.0563

2. data preprocessing

2.1. categorical encoding for 'gender' feature

In [3]:
import pandas as pd                 # import pandas for data handling

data['gender'] = data['gender'].astype('category')  # convert 'gender' values to categorical type
data['gender'] = data['gender'].cat.codes           # replace 'gender' with its numeric category codes

In [4]:
data['gender'].head()

0    1
1    1
2    1
3    0
4    1
Name: gender, dtype: int8

In [5]:
#data['height_cm'].head()

2.2. define weight frequencies for class imbalance issue for weight_kg

In [6]:
'''import pandas as pd                     # import pandas for data handling
import numpy as np                      # import numpy to help with safe division

# Assume 'data' is your DataFrame and already loaded
#print("Preview of data:\n", data.head())  # print first few rows to check data
print("\nTotal samples in dataset:", len(data))  # print total number of rows

# -----------------------------
# 1. Create boolean masks for the three weight_kg classes
# -----------------------------
class_1_mask = data['weight_kg'] < 60                      # True where weight_kg is less than 60
class_2_mask = data['weight_kg'] > 100                     # True where weight_kg is greater than 100
class_3_mask = (data['weight_kg'] >= 60) & (data['weight_kg'] <= 100)  # True where weight is between 60 and 100

# -----------------------------
# 2. Calculate class frequencies (counts)
# -----------------------------
freq_class_1 = class_1_mask.sum()          # number of samples with weight_kg < 60
freq_class_2 = class_2_mask.sum()          # number of samples with weight_kg > 100
freq_class_3 = class_3_mask.sum()          # number of samples with 60 <= weight_kg <= 100

print("\nClass frequencies:")              # header for clarity
print("Class 1 (weight_kg < 60):", freq_class_1)   # print frequency of class 1
print("Class 2 (weight_kg > 100):", freq_class_2)  # print frequency of class 2
print("Class 3 (60 <= weight_kg <= 100):", freq_class_3)  # print frequency of class 3

# -----------------------------
# 3. Number of classes according to the strategy
# -----------------------------
num_classes = 3                             # we defined three classes by the rules above
print("\nNumber of classes:", num_classes)  # print number of classes

# -----------------------------
# 4. Compute inverse-frequency weights for each class
#    Formula: w = total_samples / (num_classes * class_frequency)
# -----------------------------
total_samples = len(data)                   # total number of rows in the dataset

def safe_weight(class_freq):                # helper function to avoid division by zero
    if class_freq == 0:                     # check if a class has zero samples
        return np.nan                       # return NaN if no samples exist for that class
    return total_samples / (num_classes * class_freq)  # apply weighting formula

weight_class_1 = safe_weight(freq_class_1)  # compute weight for class 1
weight_class_2 = safe_weight(freq_class_2)  # compute weight for class 2
weight_class_3 = safe_weight(freq_class_3)  # compute weight for class 3

print("\nClass weights (inverse frequency):")          # header for class weights
print("Weight for Class 1 (weight_kg < 60):", weight_class_1)   # print weight of class 1
print("Weight for Class 2 (weight_kg > 100):", weight_class_2)  # print weight of class 2
print("Weight for Class 3 (60 <= weight_kg <= 100):", weight_class_3)  # print weight of class 3
'''

'import pandas as pd                     # import pandas for data handling\nimport numpy as np                      # import numpy to help with safe division\n\n# Assume \'data\' is your DataFrame and already loaded\n#print("Preview of data:\n", data.head())  # print first few rows to check data\nprint("\nTotal samples in dataset:", len(data))  # print total number of rows\n\n# -----------------------------\n# 1. Create boolean masks for the three weight_kg classes\n# -----------------------------\nclass_1_mask = data[\'weight_kg\'] < 60                      # True where weight_kg is less than 60\nclass_2_mask = data[\'weight_kg\'] > 100                     # True where weight_kg is greater than 100\nclass_3_mask = (data[\'weight_kg\'] >= 60) & (data[\'weight_kg\'] <= 100)  # True where weight is between 60 and 100\n\n# -----------------------------\n# 2. Calculate class frequencies (counts)\n# -----------------------------\nfreq_class_1 = class_1_mask.sum()          # number of sample

2.3. define weight frequencies for class imbalance issue for gender feature

In [7]:
'''import numpy as np                                      # import numpy for numeric utilities (like NaN)

print("Preview of gender column:\n", data['gender'].head())  # show first few gender values to inspect

# -----------------------------------
# 1. Calculate class frequencies for gender
# -----------------------------------
gender_counts = data['gender'].value_counts()           # count how many samples belong to each gender class

print("\nClass frequencies for gender:")                # header for class frequency output
for gender_class, freq in gender_counts.items():        # loop over each gender class and its frequency
    print(f"Class {gender_class}: {freq}")              # print the class label and its frequency

# -----------------------------------
# 2. Number of gender classes
# -----------------------------------
num_gender_classes = len(gender_counts)                 # compute how many distinct gender classes we have
print("\nNumber of gender classes:", num_gender_classes)  # print number of gender classes

# -----------------------------------
# 3. Compute inverse-frequency weights for each gender class
#    Formula: w = total_samples / (num_classes * class_frequency)
# -----------------------------------
total_samples = len(data)                               # total number of samples in the dataset

def safe_weight(class_freq):                            # define helper function to compute class weight safely
    if class_freq == 0:                                 # check for zero frequency to avoid division by zero
        return np.nan                                   # return NaN if a class somehow has zero samples
    return total_samples / (num_gender_classes * class_freq)  # apply the inverse-frequency weight formula

gender_weights = {}                                     # create an empty dictionary to store weights per class
for gender_class, freq in gender_counts.items():        # loop through each gender class and its frequency
    gender_weights[gender_class] = safe_weight(freq)    # compute and store the weight for this gender class

print("\nClass weights (inverse frequency) for gender:")  # header for weight output
for gender_class, weight in gender_weights.items():     # loop over each class and its weight
    print(f"Weight for class {gender_class}: {weight}") # print the computed weight for this gender class
'''

'import numpy as np                                      # import numpy for numeric utilities (like NaN)\n\nprint("Preview of gender column:\n", data[\'gender\'].head())  # show first few gender values to inspect\n\n# -----------------------------------\n# 1. Calculate class frequencies for gender\n# -----------------------------------\ngender_counts = data[\'gender\'].value_counts()           # count how many samples belong to each gender class\n\nprint("\nClass frequencies for gender:")                # header for class frequency output\nfor gender_class, freq in gender_counts.items():        # loop over each gender class and its frequency\n    print(f"Class {gender_class}: {freq}")              # print the class label and its frequency\n\n# -----------------------------------\n# 2. Number of gender classes\n# -----------------------------------\nnum_gender_classes = len(gender_counts)                 # compute how many distinct gender classes we have\nprint("\nNumber of gender class

2.4. weight frequencies for weight classes and gender classes

In [8]:
'''import numpy as np   # import numpy for numeric operations

# -------------------------------------------------
# 1. Store the already-computed weights for weight classes
#    (use the variables you created when handling weight_kg)
# -------------------------------------------------
weight_class_weights = {                          # dictionary to hold weight-class weights
    'weight_<60':  weight_class_1,                # weight for class: weight_kg < 60
    'weight_>100': weight_class_2,                # weight for class: weight_kg > 100
    'weight_60_100': weight_class_3               # weight for class: 60 <= weight_kg <= 100
}

print("Weight-class weights:", weight_class_weights)  # print weight-class weights to check

# gender_weights dict is assumed from previous step, e.g. {0: w0, 1: w1}
print("Gender-class weights:", gender_weights)        # print gender-class weights to check

# -------------------------------------------------
# 2. Multiply each gender class with each weight class
#    wi = w_weight * w_gender
# -------------------------------------------------
combined_weights = {}                                # dictionary to store combined class weights

print("\nCombined weights for each (weight_class, gender_class):")  # header
for w_label, w_w in weight_class_weights.items():    # loop over weight classes
    for g_label, w_g in gender_weights.items():      # loop over gender classes
        wi = w_w * w_g                               # multiply weight and gender class weights
        combined_weights[(w_label, g_label)] = wi    # store in dictionary
        print(f"{w_label} & gender {g_label}: {wi}") # print each combination'''

'import numpy as np   # import numpy for numeric operations\n\n# -------------------------------------------------\n# 1. Store the already-computed weights for weight classes\n#    (use the variables you created when handling weight_kg)\n# -------------------------------------------------\nweight_class_weights = {                          # dictionary to hold weight-class weights\n    \'weight_<60\':  weight_class_1,                # weight for class: weight_kg < 60\n    \'weight_>100\': weight_class_2,                # weight for class: weight_kg > 100\n    \'weight_60_100\': weight_class_3               # weight for class: 60 <= weight_kg <= 100\n}\n\nprint("Weight-class weights:", weight_class_weights)  # print weight-class weights to check\n\n# gender_weights dict is assumed from previous step, e.g. {0: w0, 1: w1}\nprint("Gender-class weights:", gender_weights)        # print gender-class weights to check\n\n# -------------------------------------------------\n# 2. Multiply each ge

2.5. create a dictionary for weights and row index

In [9]:
'''# Check current columns in the DataFrame
print("Columns before adding index column:\n", data.columns)

# Add a new column named 'index' with values from 0 to number_of_rows-1
data['index'] = range(len(data))

# Move 'index' to the front (optional, just for nicer viewing)
cols = ['index'] + [c for c in data.columns if c != 'index']  # build new column order
data = data[cols]                                            # reorder columns

# Show first few rows to verify the new indexing column
#print("\nDataFrame after adding 'index' column:\n", data.head())
'''

'# Check current columns in the DataFrame\nprint("Columns before adding index column:\n", data.columns)\n\n# Add a new column named \'index\' with values from 0 to number_of_rows-1\ndata[\'index\'] = range(len(data))\n\n# Move \'index\' to the front (optional, just for nicer viewing)\ncols = [\'index\'] + [c for c in data.columns if c != \'index\']  # build new column order\ndata = data[cols]                                            # reorder columns\n\n# Show first few rows to verify the new indexing column\n#print("\nDataFrame after adding \'index\' column:\n", data.head())\n'

In [10]:
'''import numpy as np               # import numpy for numeric operations
import pickle                    # import pickle to save Python objects

# -------------------------------------------------
# 0. We assume these already exist:
#    - weight_class_1, weight_class_2, weight_class_3
#    - gender_weights   (dict: {gender_class: weight})
# -------------------------------------------------

# create a dictionary of weight-class weights (same as before)
weight_class_weights = {         # dictionary mapping weight class labels to their weights
    'weight_<60':  weight_class_1,      # weight for class: weight_kg < 60
    'weight_>100': weight_class_2,      # weight for class: weight_kg > 100
    'weight_60_100': weight_class_3     # weight for class: 60 <= weight_kg <= 100
}

print("Weight-class weights:", weight_class_weights)  # print weight-class weights
print("Gender-class weights:", gender_weights)        # print gender-class weights

# -------------------------------------------------
# 1. Helper function to get the weight class label for a given weight_kg
# -------------------------------------------------
def get_weight_class(w):         # define a function that receives a single weight value
    if w < 60:                   # check if weight is less than 60
        return 'weight_<60'      # return label for class 1
    elif w > 100:                # check if weight is greater than 100
        return 'weight_>100'     # return label for class 2
    else:                        # otherwise weight is between 60 and 100 (inclusive)
        return 'weight_60_100'   # return label for class 3

# -------------------------------------------------
# 2. Build dictionary: keys = index values, values = combined weights
# -------------------------------------------------
final_weights = {}               # create empty dictionary to store final weights

print("\nBuilding final_weights dictionary...")  # message to track progress

for _, row in data.iterrows():   # loop over each row of the DataFrame
    idx_val = row['index']       # get the value from the 'index' column for this row
    gender_val = row['gender']   # get the gender class value for this row
    weight_val = row['weight_kg']# get the weight_kg value for this row

    w_class = get_weight_class(weight_val)        # determine weight class label from weight_kg
    w_weight = weight_class_weights[w_class]      # look up the weight-class weight
    w_gender = gender_weights[gender_val]         # look up the gender-class weight

    combined_w = w_weight * w_gender             # multiply to get combined weight w_i
    final_weights[idx_val] = combined_w          # store combined weight in dictionary with key=index

print("Number of entries in final_weights:", len(final_weights))  # print number of entries
print("First 5 items in final_weights:", list(final_weights.items())[:5])  # show first few items

# -------------------------------------------------
# 3. Check index 0: gender, weight_kg, and combined weight
# -------------------------------------------------
print("\nChecking entry with index 0...")        # message to show what we're doing

row0 = data.loc[data['index'] == 0].iloc[0]      # select the row where 'index' column equals 0

gender0 = row0['gender']                         # get gender value for index 0
weight0 = row0['weight_kg']                      # get weight_kg value for index 0
w_class0 = get_weight_class(weight0)             # get weight class label for index 0

w_weight0 = weight_class_weights[w_class0]       # get weight-class weight for index 0
w_gender0 = gender_weights[gender0]              # get gender-class weight for index 0
combined0_calc = w_weight0 * w_gender0           # calculate combined weight for index 0

print("Row 0 -> gender:", gender0)               # print gender class for index 0
print("Row 0 -> weight_kg:", weight0)            # print weight_kg for index 0
print("Row 0 -> weight class:", w_class0)        # print weight class label for index 0
print("w_weight for row 0:", w_weight0)          # print weight-class weight for index 0
print("w_gender for row 0:", w_gender0)          # print gender-class weight for index 0
print("Combined weight (calculated):", combined0_calc)        # print calculated combined weight
print("Combined weight from final_weights[0]:", final_weights[0])  # print value from dictionary

# -------------------------------------------------
# 4. Save final_weights dictionary as a pickle file
# -------------------------------------------------
print("\nSaving final_weights dictionary as pickle file...")   # message to track saving step

with open('final_weights.pkl', 'wb') as f:       # open a file named 'final_weights.pkl' in binary write mode
    pickle.dump(final_weights, f)                # write dictionary to the file using pickle

print("Dictionary saved to 'final_weights.pkl'.")# confirmation message
'''

'import numpy as np               # import numpy for numeric operations\nimport pickle                    # import pickle to save Python objects\n\n# -------------------------------------------------\n# 0. We assume these already exist:\n#    - weight_class_1, weight_class_2, weight_class_3\n#    - gender_weights   (dict: {gender_class: weight})\n# -------------------------------------------------\n\n# create a dictionary of weight-class weights (same as before)\nweight_class_weights = {         # dictionary mapping weight class labels to their weights\n    \'weight_<60\':  weight_class_1,      # weight for class: weight_kg < 60\n    \'weight_>100\': weight_class_2,      # weight for class: weight_kg > 100\n    \'weight_60_100\': weight_class_3     # weight for class: 60 <= weight_kg <= 100\n}\n\nprint("Weight-class weights:", weight_class_weights)  # print weight-class weights\nprint("Gender-class weights:", gender_weights)        # print gender-class weights\n\n# --------------------

2.6. apply stnadard sclaer for body measurements and robust scaler for cnn extracted features

In [11]:
from sklearn.preprocessing import StandardScaler, RobustScaler

# -----------------------------
# 1. Columns
# -----------------------------

# columns to exclude from any scaling
exclude_cols = ['photo_id', 'subject_id', 'index', 'gender']

# target columns (predicted outputs)
target_cols = [
    'ankle', 'arm-length', 'bicep', 'calf', 'chest', 'forearm', 'hip',
    'leg-length', 'shoulder-breadth', 'shoulder-to-crotch', 'thigh',
    'waist', 'wrist', 'weight_kg'
]

# feature columns that must use StandardScaler (but are NOT targets)
standard_feature_cols = ['height_cm']

# keep only existing columns
target_cols = [c for c in target_cols if c in data.columns]
standard_feature_cols = [c for c in standard_feature_cols if c in data.columns]

# remaining feature columns → RobustScaler
robust_feature_cols = [
    col for col in data.columns
    if col not in exclude_cols
    and col not in target_cols
    and col not in standard_feature_cols
]

# -----------------------------
# 2. Create scalers
# -----------------------------

scaler_targets = StandardScaler()           # for inverse scaling
scaler_standard_features = StandardScaler()
scaler_robust_features = RobustScaler()

# -----------------------------
# 3. Fit & transform
# -----------------------------

data[target_cols] = scaler_targets.fit_transform(data[target_cols])
data[standard_feature_cols] = scaler_standard_features.fit_transform(
    data[standard_feature_cols]
)
data[robust_feature_cols] = scaler_robust_features.fit_transform(
    data[robust_feature_cols]
)


3. model validation

3.1. split the data for independent and dependent features

In [12]:
# List of columns to be used as dependent (target) features
target_cols = [
    'ankle', 'arm-length', 'bicep', 'calf', 'chest', 'forearm', 'hip',
    'leg-length', 'shoulder-breadth', 'shoulder-to-crotch', 'thigh',
    'waist', 'wrist', 'weight_kg'
]

# Select these columns from the DataFrame as the multi-target Y
Y = data[target_cols]                  # Y will hold all dependent variables for multi-target regression

print("Selected target columns:", target_cols)  # print which columns are used as targets
print("Shape of Y (samples, targets):", Y.shape)  # print shape to confirm dimensions

Selected target columns: ['ankle', 'arm-length', 'bicep', 'calf', 'chest', 'forearm', 'hip', 'leg-length', 'shoulder-breadth', 'shoulder-to-crotch', 'thigh', 'waist', 'wrist', 'weight_kg']
Shape of Y (samples, targets): (1684, 14)


In [13]:
# Columns to drop for building independent features (X)
drop_cols = ['photo_id', 'subject_id'] + target_cols   # combine ID columns with target columns

print("Columns to drop for X:\n", drop_cols)           # show which columns will be removed

# Create X by dropping ID columns and all target columns
X = data.drop(columns=drop_cols)                       # drop the unwanted columns to get independent features

print("\nShape of X (samples, independent features):", X.shape)  # print shape of X
#print("\nColumns in X:\n", X.columns.tolist())         # list all feature names in X

Columns to drop for X:
 ['photo_id', 'subject_id', 'ankle', 'arm-length', 'bicep', 'calf', 'chest', 'forearm', 'hip', 'leg-length', 'shoulder-breadth', 'shoulder-to-crotch', 'thigh', 'waist', 'wrist', 'weight_kg']

Shape of X (samples, independent features): (1684, 5122)


In [14]:
X.head()

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,s2553,s2554,s2555,s2556,s2557,s2558,s2559,s2560,gender,height_cm
0,0.774462,1.040661,0.342109,1.366849,-0.398465,1.118232,-2.134066,-0.676307,-1.995643,0.548878,...,1.283080,0.115487,-0.576038,-0.817925,0.264697,-0.373763,-0.786373,0.623649,1,1.066025
1,1.162176,-0.619511,-0.222455,0.380021,-0.696737,-0.080061,-0.764886,0.635084,0.058381,-0.636982,...,1.388174,-0.218102,0.647752,0.725138,0.020168,-0.598226,3.017102,-0.008240,1,2.083311
2,0.443413,-0.459284,-0.377604,0.321320,-0.444318,-0.057285,-0.431309,0.158932,-0.288579,-1.554653,...,2.352519,-0.128612,-0.894785,-0.668226,0.509909,-0.659391,0.624502,1.260808,1,1.031734
3,-1.114722,0.282268,-0.369572,-0.755762,-0.090680,-0.491214,0.744916,-0.327898,0.902881,0.414726,...,0.496970,1.032414,0.211070,-0.479480,-0.684825,0.051598,-0.377326,0.769187,0,-0.425615
4,1.005786,0.823773,-0.632466,0.114854,0.365689,0.167260,-1.480106,-1.080566,-0.435842,-0.041774,...,1.103295,-1.426363,0.570939,0.856161,3.154351,0.484065,1.333055,-0.715199,1,0.288772


3.2. load the model for inferencing

In [15]:
# =============================================================================
# FT-TRANSFORMER MODEL LOADER (single script)
# - Defines required custom layers (FeatureTokenizer, FTTransformerBlock)
# - Loads your saved model (.keras or .h5)
# - Prints summary
# =============================================================================

import tensorflow as tf                              # tensorflow
from tensorflow import keras                         # keras
from tensorflow.keras import layers                  # layers

# -------------------------
# Optional: force CPU only (prevents CUDA init noise)
# -------------------------
try:
    tf.config.set_visible_devices([], "GPU")         # hide GPU devices
    print("✓ GPU hidden -> using CPU")               # log
except Exception as e:
    print("⚠️ Could not change visible devices:", e) # log

print("✓ TensorFlow version:", tf.__version__)       # log


# =============================================================================
# Custom Layers (MUST exist before loading)
# =============================================================================

@tf.keras.utils.register_keras_serializable()
class FeatureTokenizer(layers.Layer):
    """
    FT-Transformer Feature Tokenizer (memory-safe):
    - Compresses numeric features -> K numeric tokens
    - Tokenizes numeric tokens as: z_k * W_k + b_k
    - Tokenizes categorical as: Embedding(cat) + bias
    - Prepends learnable [CLS] token
    """
    def __init__(self, num_numeric_features, categorical_cardinalities, embed_dim, numeric_token_count=32, **kwargs):
        super().__init__(**kwargs)                                      # init base layer
        self.num_numeric_features = num_numeric_features                # numeric feature count
        self.categorical_cardinalities = categorical_cardinalities      # list of category sizes
        self.embed_dim = embed_dim                                      # token dim
        self.numeric_token_count = numeric_token_count                  # K numeric tokens
        self.num_categorical_features = len(categorical_cardinalities)  # number of categorical features

        # One embedding table per categorical feature
        self.cat_embeddings = []                                        # store embeddings
        for i, card in enumerate(categorical_cardinalities):            # loop categorical feats
            self.cat_embeddings.append(
                layers.Embedding(
                    input_dim=card,                                     # vocab size
                    output_dim=embed_dim,                                # embedding dim
                    name=f"cat_embedding_{i}"                            # unique name
                )
            )

        # Compress numeric features from (n_num) -> (K)
        self.numeric_compressor = layers.Dense(
            units=numeric_token_count,                                   # output K
            use_bias=True,                                               # bias
            name="numeric_compressor"                                    # name
        )

    def build(self, input_shape):
        # Tokenization weights for numeric tokens (K, D)
        self.W_num = self.add_weight(
            name="W_num_tok",                                           # name
            shape=(self.numeric_token_count, self.embed_dim),           # (K, D)
            initializer="random_normal",                                # init
            trainable=True                                              # trainable
        )

        # Tokenization bias for numeric tokens (K, D)
        self.b_num = self.add_weight(
            name="b_num_tok",                                           # name
            shape=(self.numeric_token_count, self.embed_dim),           # (K, D)
            initializer="zeros",                                        # init
            trainable=True                                              # trainable
        )

        # Bias for categorical tokens (n_cat, D)
        self.b_cat = self.add_weight(
            name="b_cat",                                               # name
            shape=(self.num_categorical_features, self.embed_dim),      # (n_cat, D)
            initializer="zeros",                                        # init
            trainable=True                                              # trainable
        )

        # Learnable CLS token (1, 1, D)
        self.cls_token = self.add_weight(
            name="cls_token",                                           # name
            shape=(1, 1, self.embed_dim),                               # (1,1,D)
            initializer="random_normal",                                # init
            trainable=True                                              # trainable
        )

        super().build(input_shape)                                      # finalize build

    def call(self, inputs, training=False):
        categorical_inputs, continuous_inputs = inputs                  # unpack
        batch_size = tf.shape(continuous_inputs)[0]                     # batch size

        # ---- numeric: compress -> tokenize ----
        z = self.numeric_compressor(continuous_inputs)                  # (B, K)
        z = tf.expand_dims(z, axis=-1)                                  # (B, K, 1)
        W = tf.expand_dims(self.W_num, axis=0)                          # (1, K, D)
        b = tf.expand_dims(self.b_num, axis=0)                          # (1, K, D)
        num_tokens = z * W + b                                          # (B, K, D)

        # ---- categorical: embedding + bias ----
        cat_tokens_list = []                                            # store cat tokens
        for i in range(self.num_categorical_features):                  # loop cat feats
            cat_i = categorical_inputs[:, i:i+1]                        # (B,1)
            emb_i = self.cat_embeddings[i](cat_i)                       # (B,1,D)
            bias_i = tf.reshape(self.b_cat[i], (1, 1, self.embed_dim))  # (1,1,D)
            cat_tokens_list.append(emb_i + bias_i)                      # add bias

        cat_tokens = tf.concat(cat_tokens_list, axis=1) if self.num_categorical_features > 0 else None  # (B,n_cat,D)

        # ---- CLS token ----
        cls = tf.tile(self.cls_token, [batch_size, 1, 1])               # (B,1,D)

        # ---- final token sequence ----
        if cat_tokens is not None:
            tokens = tf.concat([cls, num_tokens, cat_tokens], axis=1)   # (B, 1+K+n_cat, D)
        else:
            tokens = tf.concat([cls, num_tokens], axis=1)               # (B, 1+K, D)

        return tokens                                                   # return tokens

    def get_config(self):
        config = super().get_config()                                   # base config
        config.update({
            "num_numeric_features": self.num_numeric_features,          # save
            "categorical_cardinalities": self.categorical_cardinalities,# save
            "embed_dim": self.embed_dim,                                # save
            "numeric_token_count": self.numeric_token_count             # save
        })
        return config


@tf.keras.utils.register_keras_serializable()
class FTTransformerBlock(layers.Layer):
    """
    FT-Transformer Block:
    - PreNorm + residual
    - FT tweak: optionally skip LN before attention in first block
    """
    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1, attention_dropout=0.1, use_attention_norm=True, **kwargs):
        super().__init__(**kwargs)                                      # init base
        self.embed_dim = embed_dim                                      # token dim
        self.num_heads = num_heads                                      # heads
        self.ff_dim = ff_dim                                            # FF dim
        self.dropout_rate = dropout_rate                                # dropout
        self.attention_dropout = attention_dropout                      # attn dropout
        self.use_attention_norm = use_attention_norm                    # tweak flag

        projection_dim = embed_dim // num_heads                         # per-head dim

        self.ln1 = layers.LayerNormalization(epsilon=1e-6, name="ln1")   # LN before attn
        self.mha = layers.MultiHeadAttention(
            num_heads=num_heads,                                        # heads
            key_dim=projection_dim,                                     # key dim
            dropout=attention_dropout,                                  # attn dropout
            name="mha"                                                  # name
        )
        self.drop1 = layers.Dropout(dropout_rate, name="drop1")         # dropout

        self.ln2 = layers.LayerNormalization(epsilon=1e-6, name="ln2")   # LN before FFN
        self.dense1 = layers.Dense(ff_dim, activation="relu", name="ffn_dense1")  # FFN hidden
        self.drop2 = layers.Dropout(dropout_rate, name="drop2")         # dropout
        self.dense2 = layers.Dense(embed_dim, name="ffn_dense2")        # back to D
        self.drop3 = layers.Dropout(dropout_rate, name="drop3")         # dropout

    def build(self, input_shape):
        # input_shape: (B, L, D)
        self.ln1.build(input_shape)                                     # build ln1
        self.ln2.build(input_shape)                                     # build ln2
        self.dense1.build(input_shape)                                  # build dense1
        self.dense2.build((input_shape[0], input_shape[1], self.ff_dim))# build dense2
        super().build(input_shape)                                      # finalize

    def call(self, x, training=False):
        # ---- attention ----
        x_norm = self.ln1(x) if self.use_attention_norm else x          # apply LN or skip
        attn_out = self.mha(query=x_norm, key=x_norm, value=x_norm, training=training)  # self-attn
        attn_out = self.drop1(attn_out, training=training)              # dropout
        x = x + attn_out                                                # residual

        # ---- FFN ----
        x_norm2 = self.ln2(x)                                           # LN
        ffn_out = self.dense1(x_norm2)                                  # FFN
        ffn_out = self.drop2(ffn_out, training=training)                # dropout
        ffn_out = self.dense2(ffn_out)                                  # project
        ffn_out = self.drop3(ffn_out, training=training)                # dropout
        x = x + ffn_out                                                 # residual

        return x                                                        # output

    def get_config(self):
        config = super().get_config()                                   # base config
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "ff_dim": self.ff_dim,
            "dropout_rate": self.dropout_rate,
            "attention_dropout": self.attention_dropout,
            "use_attention_norm": self.use_attention_norm
        })
        return config


print("✓ Custom layers registered (FeatureTokenizer, FTTransformerBlock)")  # log


# =============================================================================
# Load Model
# =============================================================================

MODEL_PATH = "ft-transformerBest-v20.keras"  # <-- change if needed (e.g., "fttransformerBest.keras" or ".h5")
print("Loading model from:", MODEL_PATH)  # log

best_model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        "FeatureTokenizer": FeatureTokenizer,
        "FTTransformerBlock": FTTransformerBlock
    }
)

print("✅ Model loaded successfully!")     # log
best_model.summary()                      # show model

2026-02-13 15:47:34.405590: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


✓ GPU hidden -> using CPU
✓ TensorFlow version: 2.20.0
✓ Custom layers registered (FeatureTokenizer, FTTransformerBlock)
Loading model from: ft-transformerBest-v19.keras
✅ Model loaded successfully!


/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 51 variables whereas the saved optimizer has 100 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "FTTransformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ categorical_input   │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ continuous_input    │ (None, 5121)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_tokenizer   │ (None, 34, 32)    │    166,080 │ categorical_inpu… │
│ (FeatureTokenizer)  │                   │            │ continuous_input… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ft_block_1          │ (None, 34, 32)    │     12,704 │ feature_tokenize… │
│ (FTTransformerBloc… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ft_block_2          │ (None, 34, 32)    │     12,704 │ ft_block_1[0][0]  │
│ (FTTransformerBloc… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 32)        │          0 │ ft_block_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp_dense_1 (Dense) │ (None, 256)       │      8,448 │ get_item[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp_bn_1            │ (None, 256)       │      1,024 │ mlp_dense_1[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp_drop_1          │ (None, 256)       │          0 │ mlp_bn_1[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp_dense_2 (Dense) │ (None, 128)       │     32,896 │ mlp_drop_1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp_bn_2            │ (None, 128)       │        512 │ mlp_dense_2[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp_drop_2          │ (None, 128)       │          0 │ mlp_bn_2[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 14)        │      1,806 │ mlp_drop_2[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 471,582 (1.80 MB)

 Trainable params: 235,406 (919.55 KB)

 Non-trainable params: 768 (3.00 KB)

 Optimizer params: 235,408 (919.57 KB)

In [16]:
best_model.summary()

Model: "FTTransformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ categorical_input   │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ continuous_input    │ (None, 5121)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_tokenizer   │ (None, 34, 32)    │    166,080 │ categorical_inpu… │
│ (FeatureTokenizer)  │                   │            │ continuous_input… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ft_block_1          │ (None, 34, 32)    │     12,704 │ feature_tokenize… │
│ (FTTransformerBloc… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ft_block_2          │ (None, 34, 32)    │     12,704 │ ft_block_1[0][0]  │
│ (FTTransformerBloc… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 32)        │          0 │ ft_block_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp_dense_1 (Dense) │ (None, 256)       │      8,448 │ get_item[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp_bn_1            │ (None, 256)       │      1,024 │ mlp_dense_1[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp_drop_1          │ (None, 256)       │          0 │ mlp_bn_1[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp_dense_2 (Dense) │ (None, 128)       │     32,896 │ mlp_drop_1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp_bn_2            │ (None, 128)       │        512 │ mlp_dense_2[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp_drop_2          │ (None, 128)       │          0 │ mlp_bn_2[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 14)        │      1,806 │ mlp_drop_2[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 471,582 (1.80 MB)

 Trainable params: 235,406 (919.55 KB)

 Non-trainable params: 768 (3.00 KB)

 Optimizer params: 235,408 (919.57 KB)

3.3. calculate performance matrices

In [17]:
# -------------------------------------------
# 1. Imports
# -------------------------------------------
import numpy as np                               # numerical operations
import pandas as pd                              # to build a nice results table
import tensorflow as tf                          # to load and run the Keras model
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error  # metrics

# -------------------------------------------
# 3. Prepare validation data
#    We assume:
#      - X : pandas DataFrame with validation independent features (already scaled)
#      - Y : pandas DataFrame with validation targets (same order as training, already scaled)
# -------------------------------------------

print("Shape of validation X:", X.shape)            # show shape of validation features
print("Shape of validation Y:", Y.shape)            # show shape of validation targets

# Save target column names (body measurement names)
target_cols = list(Y.columns)                       # list of body measurement names
print("\nTarget columns (body measurements):")
print(target_cols)                                  # print body measurement names

# Convert X and Y to NumPy arrays for prediction / metric computation
X_val = X.values.astype("float32")                  # features as float32 array
Y_val = Y.values.astype("float32")                  # targets as float32 array

Shape of validation X: (1684, 5122)
Shape of validation Y: (1684, 14)

Target columns (body measurements):
['ankle', 'arm-length', 'bicep', 'calf', 'chest', 'forearm', 'hip', 'leg-length', 'shoulder-breadth', 'shoulder-to-crotch', 'thigh', 'waist', 'wrist', 'weight_kg']


In [18]:
# -------------------------------------------
# 4. Run inference to get predictions
# -------------------------------------------

# Split validation features exactly like training
categorical_cols = ['gender']
continuous_cols = [col for col in X.columns if col not in categorical_cols]

X_val_cat = X[categorical_cols].values.astype('int32')
X_val_cont = X[continuous_cols].values.astype('float32')

print("Categorical input shape:", X_val_cat.shape)
print("Continuous input shape:", X_val_cont.shape)

print("\nRunning inference on validation data ...")

# IMPORTANT: Pass inputs as a list (same format as training)
Y_pred = best_model.predict([X_val_cat, X_val_cont], verbose=0)

print("Inference completed.")
print("Predictions shape:", Y_pred.shape)

Categorical input shape: (1684, 1)
Continuous input shape: (1684, 5121)

Running inference on validation data ...
Inference completed.
Predictions shape: (1684, 14)


In [19]:
'''# -------------------------------------------
# 4.1 Inverse-transform targets to original units
#      Assumes scaler_targets was fitted on target_cols during preprocessing
# -------------------------------------------

# Y_val and Y_pred are currently in [-1, 1] scaled space
# Convert both back to original measurement units (cm, kg, etc.)
Y_true_orig = scaler_targets.inverse_transform(Y_val)
Y_pred_orig = scaler_targets.inverse_transform(Y_pred)'''

'# -------------------------------------------\n# 4.1 Inverse-transform targets to original units\n#      Assumes scaler_targets was fitted on target_cols during preprocessing\n# -------------------------------------------\n\n# Y_val and Y_pred are currently in [-1, 1] scaled space\n# Convert both back to original measurement units (cm, kg, etc.)\nY_true_orig = scaler_targets.inverse_transform(Y_val)\nY_pred_orig = scaler_targets.inverse_transform(Y_pred)'

In [20]:

# -------------------------------------------
# 5. Compute metrics per body measurement
# -------------------------------------------
print("\nComputing metrics (R^2, MSE, MAE) for each body measurement ...")

results = []                                        # list to collect metric rows

# Loop over each target dimension / body measurement
for i, name in enumerate(target_cols):              # i = column index, name = column name
    y_true = Y_val[:, i]                            # true values for this measurement
    y_pred = Y_pred[:, i]                           # predicted values for this measurement

    r2  = r2_score(y_true, y_pred)                  # compute R^2 score
    mse = mean_squared_error(y_true, y_pred)        # compute mean squared error
    mae = mean_absolute_error(y_true, y_pred)       # compute mean absolute error

    # append metrics as a dict (one row)
    results.append({
        "body_measurement": name,                   # column for measurement name
        "r2": r2,                                   # R^2 value
        "mse": mse,                                 # MSE value
        "mae": mae                                  # MAE value
    })

    # print quick summary for this measurement
    print(f"{name:20s} -> R^2: {r2:.4f}, MSE: {mse:.6f}, MAE: {mae:.6f}")

# Convert results list to a DataFrame for nice tabular view
results_df = pd.DataFrame(results)                  # create DataFrame from list of dicts

print("\nPer-measurement metrics table:")
print(results_df)                                   # display table with all metrics



Computing metrics (R^2, MSE, MAE) for each body measurement ...
ankle                -> R^2: 0.0732, MSE: 0.926789, MAE: 0.761170
arm-length           -> R^2: -0.1954, MSE: 1.195373, MAE: 0.851900
bicep                -> R^2: -0.1401, MSE: 1.140126, MAE: 0.890728
calf                 -> R^2: 0.0624, MSE: 0.937629, MAE: 0.806418
chest                -> R^2: 0.3316, MSE: 0.668375, MAE: 0.628486
forearm              -> R^2: 0.4013, MSE: 0.598659, MAE: 0.628034
hip                  -> R^2: 0.0418, MSE: 0.958156, MAE: 0.797303
leg-length           -> R^2: -0.0598, MSE: 1.059824, MAE: 0.793698
shoulder-breadth     -> R^2: 0.3133, MSE: 0.686750, MAE: 0.652302
shoulder-to-crotch   -> R^2: 0.1830, MSE: 0.817028, MAE: 0.734280
thigh                -> R^2: -0.3021, MSE: 1.302119, MAE: 0.935684
waist                -> R^2: -0.0295, MSE: 1.029471, MAE: 0.845982
wrist                -> R^2: 0.1699, MSE: 0.830109, MAE: 0.733353
weight_kg            -> R^2: 0.1006, MSE: 0.899359, MAE: 0.716050

Per-m

In [21]:
# -------------------------------------------
# 6. Compute overall (mean) scores across all measurements
# -------------------------------------------
overall_r2  = results_df["r2"].mean()               # mean R^2 over all body measurements
overall_mse = results_df["mse"].mean()              # mean MSE over all body measurements
overall_mae = results_df["mae"].mean()              # mean MAE over all body measurements

print("\nOverall (mean) scores across all body measurements:")
print(f"Mean R^2 : {overall_r2:.4f}")
print(f"Mean MSE : {overall_mse:.6f}")
print(f"Mean MAE : {overall_mae:.6f}")

# Optionally, add a final row with the overall mean scores to the table
overall_row = {
    "body_measurement": "OVERALL_MEAN",             # label row as overall
    "r2": overall_r2,
    "mse": overall_mse,
    "mae": overall_mae
}
results_df = pd.concat([results_df, pd.DataFrame([overall_row])], ignore_index=True)

print("\nMetrics table including overall mean row:")
print(results_df)                                   # final table with per-measurement + overall row



Overall (mean) scores across all body measurements:
Mean R^2 : 0.0679
Mean MSE : 0.932126
Mean MAE : 0.769671

Metrics table including overall mean row:
      body_measurement        r2       mse       mae
0                ankle  0.073211  0.926789  0.761170
1           arm-length -0.195373  1.195373  0.851900
2                bicep -0.140126  1.140126  0.890728
3                 calf  0.062371  0.937629  0.806418
4                chest  0.331625  0.668375  0.628486
5              forearm  0.401341  0.598659  0.628034
6                  hip  0.041844  0.958156  0.797303
7           leg-length -0.059824  1.059824  0.793698
8     shoulder-breadth  0.313250  0.686750  0.652302
9   shoulder-to-crotch  0.182972  0.817028  0.734280
10               thigh -0.302119  1.302119  0.935684
11               waist -0.029471  1.029471  0.845982
12               wrist  0.169891  0.830109  0.733353
13           weight_kg  0.100641  0.899359  0.716050
14        OVERALL_MEAN  0.067874  0.932126  0.76967

3.4. performance based on real values after inverse transforming

In [22]:
# -------------------------------------------
# 4.1 Inverse-transform targets to original units
# -------------------------------------------

Y_true_orig = scaler_targets.inverse_transform(Y_val)
Y_pred_orig = scaler_targets.inverse_transform(Y_pred)


In [23]:
# -------------------------------------------
# 5. Compute metrics per body measurement (in original units)
# -------------------------------------------
print("\nComputing metrics (R^2, MSE, MAE) for each body measurement in ORIGINAL UNITS ...")

results = []                                        # list to collect metric rows

# Loop over each target dimension / body measurement
for i, name in enumerate(target_cols):              # i = column index, name = column name
    # use inverse-transformed (real-unit) values
    y_true = Y_true_orig[:, i]                      # true values for this measurement (cm, kg, etc.)
    y_pred = Y_pred_orig[:, i]                      # predicted values for this measurement (cm, kg, etc.)

    r2  = r2_score(y_true, y_pred)                  # compute R^2 score
    mse = mean_squared_error(y_true, y_pred)        # compute mean squared error
    mae = mean_absolute_error(y_true, y_pred)       # compute mean absolute error

    # append metrics as a dict (one row)
    results.append({
        "body_measurement": name,                   # column for measurement name
        "r2": r2,                                   # R^2 value
        "mse": mse,                                 # MSE value (in squared real units)
        "mae": mae                                  # MAE value (in real units)
    })

    # print quick summary for this measurement
    print(f"{name:20s} -> R^2: {r2:.4f}, MSE: {mse:.6f}, MAE: {mae:.6f}")

# Convert results list to a DataFrame for nice tabular view
results_df = pd.DataFrame(results)                  # create DataFrame from list of dicts

print("\nPer-measurement metrics table (original units):")
print(results_df)                                   # display table with all metrics


Computing metrics (R^2, MSE, MAE) for each body measurement in ORIGINAL UNITS ...
ankle                -> R^2: 0.0732, MSE: 2.905010, MAE: 1.347613
arm-length           -> R^2: -0.1954, MSE: 10.933417, MAE: 2.576409
bicep                -> R^2: -0.1401, MSE: 15.534739, MAE: 3.287915
calf                 -> R^2: 0.0624, MSE: 8.400161, MAE: 2.413729
chest                -> R^2: 0.3316, MSE: 68.590652, MAE: 6.366756
forearm              -> R^2: 0.4013, MSE: 3.303848, MAE: 1.475379
hip                  -> R^2: 0.0418, MSE: 71.788750, MAE: 6.901343
leg-length           -> R^2: -0.0598, MSE: 24.170395, MAE: 3.790359
shoulder-breadth     -> R^2: 0.3133, MSE: 3.964114, MAE: 1.567194
shoulder-to-crotch   -> R^2: 0.1830, MSE: 15.369860, MAE: 3.184772
thigh                -> R^2: -0.3021, MSE: 37.684216, MAE: 5.033659
waist                -> R^2: -0.0295, MSE: 109.251282, MAE: 8.715000
wrist                -> R^2: 0.1699, MSE: 1.335075, MAE: 0.930033
weight_kg            -> R^2: 0.1006, MSE: 178

In [24]:
# -------------------------------------------
# 6. Compute overall (mean) scores across all measurements (original units)
# -------------------------------------------
overall_r2  = results_df["r2"].mean()               # mean R^2 over all body measurements
overall_mse = results_df["mse"].mean()              # mean MSE over all body measurements
overall_mae = results_df["mae"].mean()              # mean MAE over all body measurements

print("\nOverall (mean) scores across all body measurements (original units):")
print(f"Mean R^2 : {overall_r2:.4f}")
print(f"Mean MSE : {overall_mse:.6f}")
print(f"Mean MAE : {overall_mae:.6f}")

# Optionally, add a final row with the overall mean scores to the table
overall_row = {
    "body_measurement": "OVERALL_MEAN",             # label row as overall
    "r2": overall_r2,
    "mse": overall_mse,
    "mae": overall_mae
}
results_df = pd.concat([results_df, pd.DataFrame([overall_row])], ignore_index=True)

print("\nMetrics table including overall mean row (original units):")
print(results_df)                                   # final table with per-measurement + overall row



Overall (mean) scores across all body measurements (original units):
Mean R^2 : 0.0679
Mean MSE : 39.443747
Mean MAE : 4.120824

Metrics table including overall mean row (original units):
      body_measurement        r2         mse        mae
0                ankle  0.073212    2.905010   1.347613
1           arm-length -0.195373   10.933417   2.576409
2                bicep -0.140126   15.534739   3.287915
3                 calf  0.062371    8.400161   2.413729
4                chest  0.331625   68.590652   6.366756
5              forearm  0.401341    3.303848   1.475379
6                  hip  0.041844   71.788750   6.901343
7           leg-length -0.059824   24.170395   3.790359
8     shoulder-breadth  0.313250    3.964114   1.567194
9   shoulder-to-crotch  0.182972   15.369860   3.184772
10               thigh -0.302119   37.684216   5.033659
11               waist -0.029471  109.251282   8.715000
12               wrist  0.169891    1.335075   0.930033
13           weight_kg  0.1

4. Compare BMI

In [25]:
# -------------------------------------------
# Inverse-transform height_cm (REAL UNITS)
# -------------------------------------------

# height index in X
height_idx = X.columns.get_loc("height_cm")

# extract scaled height
height_scaled = X_val[:, height_idx].reshape(-1, 1)

# inverse transform
height_cm_orig = scaler_standard_features.inverse_transform(height_scaled).ravel()

# convert to meters
height_m_orig = height_cm_orig / 100.0


In [26]:
# -------------------------------------------
# Extract real weights (kg)
# -------------------------------------------

weight_idx = target_cols.index("weight_kg")

weight_true = Y_true_orig[:, weight_idx]
weight_pred = Y_pred_orig[:, weight_idx]


In [27]:
# -------------------------------------------
# BMI computation
# -------------------------------------------

bmi_true = weight_true / (height_m_orig ** 2)
bmi_pred = weight_pred / (height_m_orig ** 2)


In [28]:
# -------------------------------------------
# BMI MAE per record
# -------------------------------------------

bmi_mae_per_record = np.abs(bmi_true - bmi_pred)


In [29]:
average_bmi_mae = bmi_mae_per_record.mean()

print(f"Average BMI MAE across {len(bmi_mae_per_record)} records: {average_bmi_mae:.4f}")


Average BMI MAE across 1684 records: 3.3773


* MAE = 2.5 -> predictions are off (away) by about 2.5 units from the true values. Sometimes error can be above (overestimates) and sometimes error can be below (underestimated). On average model is wrong by 2.5 unots
* R Squared = 0.55 -> model explains about 55% of the variation in the overall body measurements
* MSE = 12.25 -> squared error between the predictions and true values are 12.25 units
* MSE < 0 ->  get smaller overall error by ignoring all inputs and just predicting the average hip size for everyone, instead of using your model’s predictions.
* MSE = -0.5 -> worse than the mean, with 50% more squared error than the mean